# ASR

In [1]:
import os
import gc
import torch
from pydub import AudioSegment
import nemo.collections.asr as nemo_asr

def cleanup_all():
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

asr_model = nemo_asr.models.ASRModel.from_pretrained(
    model_name="nvidia/parakeet-tdt-0.6b-v2"
)
asr_model = asr_model.to("cuda")
print("ASR model loaded.")

[NeMo W 2025-12-04 09:11:37 nemo_logging:405] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


CUDA available: True
GPU name: NVIDIA GeForce RTX 3060 Laptop GPU
[NeMo I 2025-12-04 09:11:45 nemo_logging:393] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2025-12-04 09:11:46 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 40.0
    min_duration: 0.1
    text_field: answer
    batch_duration: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2025-12-04 09:11:46 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config :

[NeMo I 2025-12-04 09:11:46 nemo_logging:393] PADDING: 0
[NeMo I 2025-12-04 09:11:50 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2025-12-04 09:11:50 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2025-12-04 09:11:50 nemo_logging:405] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: No `cuda-python` module. Please do `pip install cuda-python>=12.3`


[NeMo I 2025-12-04 09:11:50 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2025-12-04 09:11:50 nemo_logging:405] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: No `cuda-python` module. Please do `pip install cuda-python>=12.3`


[NeMo I 2025-12-04 09:11:54 nemo_logging:393] Model EncDecRNNTBPEModel was successfully restored from /home/tckleme-dev/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v2/snapshots/48b630d20b000e5ad3735e5378a2d9bde3f80826/parakeet-tdt-0.6b-v2.nemo.
ASR model loaded.


## Preprocess Audio

In [2]:
audio_path = "doc_patient_convo.mp3"
audio = AudioSegment.from_file(audio_path)

# Convert to mono, 16kHz
audio = audio.set_channels(1).set_frame_rate(16000)
processed_path = "doc_patient_convo_mono16k.wav"
audio.export(processed_path, format="wav")
print("Audio converted to mono 16kHz.")

Audio converted to mono 16kHz.


## Transcribe Entire Audio

In [3]:
os.makedirs("stream_transcripts", exist_ok=True)

output = asr_model.transcribe([processed_path], timestamps=True)
hyp = output[0]

# Get plain text
transcript_text = ""
if isinstance(hyp.timestamp, list):
    if all(isinstance(seg, str) for seg in hyp.timestamp):
        transcript_text = " ".join(hyp.timestamp)
    else:
        for seg in hyp.timestamp:
            transcript_text += seg.text + " "
else:
    transcript_text = hyp.text

transcript_file = "stream_transcripts/full_transcript.txt"
with open(transcript_file, "w", encoding="utf-8") as f:
    f.write(transcript_text.strip())

cleanup_all()
print("Full transcript saved.")

[NeMo I 2025-12-04 09:12:06 nemo_logging:393] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2025-12-04 09:12:05 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2025-12-04 09:12:05 nemo_logging:405] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: No `cuda-python` module. Please do `pip install cuda-python>=12.3`
Transcribing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:09<00:00,  9.09s/it]


Full transcript saved.


In [4]:
!nvidia-smi

Thu Dec  4 09:12:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.10              Driver Version: 581.29         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...    On  |   00000000:01:00.0  On |                  N/A |
| N/A   39C    P5             11W /   50W |    3049MiB /   6144MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Unload ASR Model to free some memory

In [5]:
print("Unloading ASR model and freeing memory...")
del asr_model
cleanup_all()
print("ASR model unloaded, GPU memory freed.")

Unloading ASR model and freeing memory...
ASR model unloaded, GPU memory freed.


## Sentence-Based Chunking

In [6]:
import re

with open(transcript_file, "r", encoding="utf-8") as f:
    all_text = f.read()

# Split by sentence
sentences = re.split(r'(?<=[.!?]) +', all_text)

# Chunk ~200 words per chunk
chunks = []
current = ""
for s in sentences:
    if len(current.split()) + len(s.split()) > 200:
        chunks.append(current.strip())
        current = ""
    current += s + " "
if current:
    chunks.append(current.strip())

print(f"{len(chunks)} chunks created for summarization.")

3 chunks created for summarization.


# Summarization

In [7]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=0 if torch.cuda.is_available() else -1
)

summaries = []
for c in chunks:
    summaries.append(summarizer(c, max_length=250, min_length=80, do_sample=False)[0]['summary_text'])

final_summary = " ".join(summaries)
print(final_summary)

Device set to use cuda:0
Your max_length is set to 250, but your input_length is only 13. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)


Zara says she's had a cough for three weeks and it just won't go away. She doesn't have a fever, chills, or body aches, but says she feels run down and tired. Doctor: Three weeks is definitely a good amount of time to see if it will pass on its own. Do you have any history of asthma or allergies? No, none that I know of. I've generally been pretty healthy. It sounds like you might have a touch of bronchitis. I'm going to prescribe you some cough medicine to help loosen the mucus. I'll also write you a prescription for an antibiotic just in case it's bacterial. Be sure to take the entire course of antibiotics, even if you start feeling better. It's important to kill all the bacteria to prevent resistance. Drink lots of fluids, especially warm liquids like tea with honey. Avoid irritants like smoke or strong perfumes. Come back in three days if you have not improved. If you have improved, come back again in a week. If not, go back to the previous page and try again. Back to the page you 